In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

DATA_PATH = "/content/drive/MyDrive/COMP472_Project/data/"

image_train = pd.read_csv(DATA_PATH + "image_train.csv")
image_test = pd.read_csv(DATA_PATH + "image_test.csv")

sentiment_train = pd.read_csv(DATA_PATH + "sentiment_train.csv")
sentiment_test = pd.read_csv(DATA_PATH + "sentiment_test.csv")

sample_submission = pd.read_csv(DATA_PATH + "sample_submission.csv")

In [ ]:
print("Image training:", image_train.shape)
print("Image test:", image_test.shape)

print("Sentiment training:", sentiment_train.shape)
print("Sentiment test:", sentiment_test.shape)

Image training: (20000, 502)
Image test: (5000, 501)
Sentiment training: (12000, 770)
Sentiment test: (5000, 769)


In [ ]:
from sklearn.model_selection import train_test_split

def split_dataset(df):
    feature_columns = [
        column for column in df.columns if column.startswith("feature_")
    ]

    X = df[feature_columns].to_numpy(dtype="float32")
    y = df["label"].to_numpy(dtype="int64")

    return train_test_split(
        X, y,
        test_size=0.20,
        random_state=42,
        stratify=y,
    )

X_img_train, X_img_val, y_img_train, y_img_val = split_dataset(image_train)
X_sent_train, X_sent_val, y_sent_train, y_sent_val = split_dataset(sentiment_train)

print("Image:", X_img_train.shape, X_img_val.shape)
print("Sentiment:", X_sent_train.shape, X_sent_val.shape)



Image: (16000, 500) (4000, 500)
Sentiment: (9600, 768) (2400, 768)


In [ ]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

def make_loaders(X_tr, y_tr, X_val, y_val, batch_size=64):
    train_dataset = TensorDataset(
        torch.tensor(X_tr, dtype=torch.float32),
        torch.tensor(y_tr, dtype=torch.long)
    )
    val_dataset = TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.long)
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

class TunedMLP(nn.Module):
    def __init__(self, input_size, output_size, hidden1=128, hidden2=None, dropout=0.3):
        super().__init__()
        if hidden2 is None:
            self.network = nn.Sequential(
                nn.Linear(input_size, hidden1),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden1, output_size)
            )
        else:
            self.network = nn.Sequential(
                nn.Linear(input_size, hidden1),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden1, hidden2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden2, output_size)
            )

    def forward(self, x):
        return self.network(x)

# train model and save weights with best val accuracy
def test_config(X_tr, y_tr, X_val, y_val, output_size, hidden1, hidden2, dropout, lr, batch_size, epochs):
    torch.manual_seed(42)
    train_loader, val_loader = make_loaders(X_tr, y_tr, X_val, y_val, batch_size=batch_size)

    model = TunedMLP(X_tr.shape[1], output_size, hidden1=hidden1, hidden2=hidden2, dropout=dropout)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_acc = 0.0
    best_epoch = 0
    best_weights = None

    for epoch in range(epochs):
        model.train()
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = correct / total
        if val_acc > best_acc:
            best_acc = val_acc
            best_epoch = epoch + 1
            best_weights = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_weights)
    return model, best_acc, best_epoch

# image nn tuning
print("=== Image NN Basic Hyperparameter Tuning ===")
image_configs = [
    {"hidden1": 256, "hidden2": None, "dropout": 0.2, "lr": 0.001, "batch_size": 64, "epochs": 30},
    {"hidden1": 512, "hidden2": None, "dropout": 0.3, "lr": 0.001, "batch_size": 64, "epochs": 30},
    {"hidden1": 512, "hidden2": 256, "dropout": 0.3, "lr": 0.001, "batch_size": 64, "epochs": 30},
]

best_img_acc = 0.0
best_img_cfg = None
for cfg in image_configs:
    _, acc, ep = test_config(X_img_train, y_img_train, X_img_val, y_img_val, output_size=50, **cfg)
    print(f"Hidden=({cfg['hidden1']}, {cfg['hidden2']}) | Drop={cfg['dropout']} | LR={cfg['lr']} | Best Val Acc: {acc:.2%} (Epoch {ep})")
    if acc > best_img_acc:
        best_img_acc = acc
        best_img_cfg = cfg

# sentiment nn tuning
print("\n=== Sentiment NN Basic Hyperparameter Tuning ===")
sentiment_configs = [
    {"hidden1": 256, "hidden2": None, "dropout": 0.5, "lr": 0.0003, "batch_size": 64, "epochs": 25},
    {"hidden1": 128, "hidden2": None, "dropout": 0.5, "lr": 0.0002, "batch_size": 32, "epochs": 25},
    {"hidden1": 64, "hidden2": None, "dropout": 0.4, "lr": 0.0003, "batch_size": 64, "epochs": 25},
    {"hidden1": 256, "hidden2": None, "dropout": 0.6, "lr": 0.0005, "batch_size": 128, "epochs": 25},
]

best_sent_acc = 0.0
best_sent_cfg = None
for cfg in sentiment_configs:
    _, acc, ep = test_config(X_sent_train, y_sent_train, X_sent_val, y_sent_val, output_size=3, **cfg)
    print(f"Hidden=({cfg['hidden1']}, {cfg['hidden2']}) | Drop={cfg['dropout']} | LR={cfg['lr']} | Batch={cfg['batch_size']} | Best Val Acc: {acc:.2%} (Epoch {ep})")
    if acc > best_sent_acc:
        best_sent_acc = acc
        best_sent_cfg = cfg

print("\n==========================================")
print(f"Best Image NN:     {best_img_acc:.2%} | Config: {best_img_cfg}")
print(f"Best Sentiment NN: {best_sent_acc:.2%} | Config: {best_sent_cfg}")
print("==========================================")

=== Image NN Basic Hyperparameter Tuning ===
Hidden=(256, None) | Drop=0.2 | LR=0.001 | Best Val Acc: 32.92% (Epoch 27)
Hidden=(512, None) | Drop=0.3 | LR=0.001 | Best Val Acc: 33.60% (Epoch 24)
Hidden=(512, 256) | Drop=0.3 | LR=0.001 | Best Val Acc: 36.40% (Epoch 30)

=== Sentiment NN Basic Hyperparameter Tuning ===
Hidden=(256, None) | Drop=0.5 | LR=0.0003 | Batch=64 | Best Val Acc: 67.50% (Epoch 16)
Hidden=(128, None) | Drop=0.5 | LR=0.0002 | Batch=32 | Best Val Acc: 67.08% (Epoch 20)
Hidden=(64, None) | Drop=0.4 | LR=0.0003 | Batch=64 | Best Val Acc: 66.71% (Epoch 17)
Hidden=(256, None) | Drop=0.6 | LR=0.0005 | Batch=128 | Best Val Acc: 67.29% (Epoch 22)

Best Image NN:     36.40% | Config: {'hidden1': 512, 'hidden2': 256, 'dropout': 0.3, 'lr': 0.001, 'batch_size': 64, 'epochs': 30}
Best Sentiment NN: 67.50% | Config: {'hidden1': 256, 'hidden2': None, 'dropout': 0.5, 'lr': 0.0003, 'batch_size': 64, 'epochs': 25}


In [ ]:
import numpy as np

def predict_knn(X_train, y_train, X_query, k=3):
    predictions = []

    for query in X_query:
        # Compute one squared distance per training row.
        differences = X_train - query
        squared_distances = (differences ** 2).sum(axis=1)

        # Find the k closest training rows and look up their labels.
        nearest_indices = np.argsort(squared_distances)[:k]
        nearest_labels = y_train[nearest_indices]

        # Count the votes and pick the label with the most votes.
        votes = np.bincount(nearest_labels)
        predicted_label = votes.argmax()
        predictions.append(predicted_label)

    return np.array(predictions)

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from google.colab import files

image_features = [c for c in image_train.columns if c.startswith("feature_")]
sentiment_features = [c for c in sentiment_train.columns if c.startswith("feature_")]

X_img_full = image_train[image_features].to_numpy(dtype=np.float32)
y_img_full = image_train["label"].to_numpy(dtype=np.int64)
X_img_test = image_test[image_features].to_numpy(dtype=np.float32)

X_sent_full = sentiment_train[sentiment_features].to_numpy(dtype=np.float32)
y_sent_full = sentiment_train["label"].to_numpy(dtype=np.int64)
X_sent_test = sentiment_test[sentiment_features].to_numpy(dtype=np.float32)

# reuse image predictions if already computed
if "image_predictions" not in globals():
    print("Predicting images with k-NN (k=1)...")
    image_predictions = predict_knn(X_img_full, y_img_full, X_img_test, k=1)

# train final sentiment NN with best config
print("Training final sentiment neural network (v3)...")
torch.manual_seed(42)

final_sentiment_v3 = nn.Sequential(
    nn.Linear(X_sent_full.shape[1], 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 3)
)

optimizer = torch.optim.Adam(final_sentiment_v3.parameters(), lr=0.0003)
criterion = nn.CrossEntropyLoss()

train_data = TensorDataset(
    torch.tensor(X_sent_full, dtype=torch.float32),
    torch.tensor(y_sent_full, dtype=torch.long)
)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

for epoch in range(16):
    final_sentiment_v3.train()
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = final_sentiment_v3(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch + 1}/16 completed")

final_sentiment_v3.eval()
with torch.no_grad():
    sentiment_predictions_v3 = final_sentiment_v3(
        torch.tensor(X_sent_test, dtype=torch.float32)
    ).argmax(dim=1).numpy()

# combine predictions and match submission order
predictions_v3 = pd.concat([
    pd.DataFrame({"id": image_test["id"], "label": image_predictions}),
    pd.DataFrame({"id": sentiment_test["id"], "label": sentiment_predictions_v3})
], ignore_index=True)

submission_v3 = sample_submission[["id"]].merge(
    predictions_v3,
    on="id",
    how="left",
    validate="one_to_one"
)

assert len(submission_v3) == 10000
assert submission_v3["label"].notna().all()
assert submission_v3["id"].is_unique

submission_v3["label"] = submission_v3["label"].astype(int)
submission_v3.to_csv("submission_v3.csv", index=False)

print("Submission v3 ready!")
print(submission_v3.shape)
print(submission_v3.head())

files.download("submission_v3.csv")

Training final sentiment neural network (v3)...
Epoch 1/16 completed
Epoch 2/16 completed
Epoch 3/16 completed
Epoch 4/16 completed
Epoch 5/16 completed
Epoch 6/16 completed
Epoch 7/16 completed
Epoch 8/16 completed
Epoch 9/16 completed
Epoch 10/16 completed
Epoch 11/16 completed
Epoch 12/16 completed
Epoch 13/16 completed
Epoch 14/16 completed
Epoch 15/16 completed
Epoch 16/16 completed
Submission v3 ready!
(10000, 2)
            id  label
0  text_003234      2
1  text_004517      2
2   img_000140     47
3   img_004880     15
4  text_002494      0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>